### 📈 More like Situational Unawareness LP

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [🛡️ Learn to Run a Personal Hedge Fund](https://quantguild.com/personal-hedge-fund)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### 📉 The mechanics of short selling

  Short selling involves borrowing shares and selling them on the open market, with the obligation to buy them back later and return them to the lender.
  
  If the price drops, the short seller profits by buying shares back at a lower price; if the price rises, losses can be unlimited since there's no cap on how high a stock can go.
  
  Going short increases portfolio leverage and exposes the trader to margin calls, lending fees, and forced buy-ins.
  
  Successful short selling requires managing these risks and being mindful of the mechanics: borrowing costs, liquidity, and the need for sufficient collateral to cover potential losses.
  
  $$
  \text{Profit from short sale} = P_{\text{sell}} - P_{\text{buy}} - \text{borrowing costs}
  $$

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Config
# ============================================================

RANDOM_SEED = 12
FRAME_DURATION = 85
FRAME_STRIDE = 2

FIG_WIDTH = 1_200
FIG_HEIGHT = 700

# Short-sale assumptions.
STARTING_CAPITAL = 50_000.0       # Cash/equity deposited by the trader.
ENTRY_PRICE = 100.0               # Price at which the short is opened.
SHARES_SHORT = 1_000              # Shares borrowed and sold short.
MAINTENANCE_MARGIN = 0.30         # Broker requires equity >= 30% of short market value.
ANNUAL_BORROW_FEE = 0.04          # 4% annual stock-borrow fee.
COVER_SLIPPAGE_BPS = 15.0         # Adverse execution when the broker liquidates.
COVER_COMMISSION = 5.0

OUTPUT_HTML = Path("/mnt/data/short_selling_margin_call_animation.html")
WRITE_HTML = True
SHOW_FIGURE = False


# ============================================================
# Styling
# ============================================================

off_white = "#e0e0e0"
muted_white = "#aaaaaa"
axis_grid = "rgba(255,255,255,0.10)"
baseline_color = "#777777"

stock_blue = "#4da3ff"
equity_green = "#29d17d"
margin_red = "#ff4d5a"
borrow_orange = "#ff9f43"
marker_white = "#f4f4f4"

axis_style = dict(
    showgrid=True,
    gridcolor=axis_grid,
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white),
)


# ============================================================
# Helpers
# ============================================================

def padded_range(values, pad_fraction=0.10, min_pad=1.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)
    return [low - pad, high + pad]


def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


def money(value: float) -> str:
    return f"${value:,.0f}"


def pct(value: float) -> str:
    return f"{100.0 * value:.1f}%"


# ============================================================
# Generate a deterministic stock path designed to teach the story:
# mild decline -> profitable short -> sharp squeeze -> margin call
# ============================================================

def generate_stock_path(seed: int = RANDOM_SEED):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2027-01-04", periods=120)
    n = len(dates)

    returns = np.zeros(n, dtype=float)

    # Phase 1: quiet trading after the short is opened.
    returns[1:25] = rng.normal(-0.0010, 0.0040, 24)

    # Phase 2: the stock falls and the short initially makes money.
    returns[25:45] = rng.normal(-0.0045, 0.0060, 20)

    # Phase 3: a squeeze begins with a gap up, then persistent buying.
    returns[45] = 0.055
    returns[46:82] = rng.normal(0.0080, 0.0080, 36)

    # Phase 4: the stock keeps moving after the broker has liquidated the short.
    returns[82:] = rng.normal(0.0020, 0.0060, n - 82)

    prices = np.zeros(n, dtype=float)
    prices[0] = ENTRY_PRICE
    for i in range(1, n):
        prices[i] = prices[i - 1] * (1.0 + returns[i])

    return {
        "dates": dates,
        "returns": returns,
        "prices": prices,
        "squeeze_start": 45,
    }


# ============================================================
# Simulate the short account
#
# While the short is open:
#   short P&L       = (entry price - current price) * shares
#   account equity  = starting capital + short P&L - borrow fees
#   margin ratio    = account equity / current short market value
#
# Forced liquidation occurs when:
#   account equity <= maintenance margin * short market value
# ============================================================

def simulate_short_account(dates, prices):
    n = len(dates)
    short_proceeds = ENTRY_PRICE * SHARES_SHORT
    daily_borrow_rate = ANNUAL_BORROW_FEE / 252.0

    borrow_fees = np.zeros(n, dtype=float)
    unrealized_pnl = np.zeros(n, dtype=float)
    equity_before_liquidation = np.zeros(n, dtype=float)
    maintenance_required = np.full(n, np.nan, dtype=float)
    margin_ratio = np.full(n, np.nan, dtype=float)
    call_price_boundary = np.full(n, np.nan, dtype=float)

    margin_call_idx = None

    for i in range(n):
        if i > 0:
            # Fee is charged on the prior day's short market value.
            borrow_fees[i] = (
                borrow_fees[i - 1]
                + daily_borrow_rate * SHARES_SHORT * prices[i - 1]
            )

        unrealized_pnl[i] = (ENTRY_PRICE - prices[i]) * SHARES_SHORT
        equity_before_liquidation[i] = (
            STARTING_CAPITAL + unrealized_pnl[i] - borrow_fees[i]
        )

        short_market_value = SHARES_SHORT * prices[i]
        maintenance_required[i] = MAINTENANCE_MARGIN * short_market_value
        margin_ratio[i] = equity_before_liquidation[i] / short_market_value

        # This is the stock price that would place the account exactly at the
        # maintenance margin boundary, including borrow fees accrued to date.
        call_price_boundary[i] = (
            STARTING_CAPITAL + short_proceeds - borrow_fees[i]
        ) / (SHARES_SHORT * (1.0 + MAINTENANCE_MARGIN))

        if (
            margin_call_idx is None
            and equity_before_liquidation[i] <= maintenance_required[i]
        ):
            margin_call_idx = i
            break

    if margin_call_idx is None:
        raise RuntimeError(
            "The generated price path did not trigger a margin call. "
            "Increase the squeeze strength or lower starting capital."
        )

    # Continue the borrow-fee and boundary arrays only through the liquidation day.
    # After that there is no short position and therefore no maintenance requirement.
    call_idx = margin_call_idx
    borrow_fees[call_idx + 1:] = borrow_fees[call_idx]
    unrealized_pnl[call_idx + 1:] = unrealized_pnl[call_idx]
    equity_before_liquidation[call_idx + 1:] = equity_before_liquidation[call_idx]

    cover_price = prices[call_idx] * (1.0 + COVER_SLIPPAGE_BPS / 10_000.0)
    realized_short_pnl = (ENTRY_PRICE - cover_price) * SHARES_SHORT
    liquidation_equity = (
        STARTING_CAPITAL
        + realized_short_pnl
        - borrow_fees[call_idx]
        - COVER_COMMISSION
    )

    # The displayed equity curve becomes flat after forced liquidation because
    # the position has been bought back and market exposure is zero.
    account_equity = equity_before_liquidation.copy()
    account_equity[call_idx:] = liquidation_equity

    # Stop these curves at the moment the short is closed.
    maintenance_required[call_idx + 1:] = np.nan
    margin_ratio[call_idx + 1:] = np.nan
    call_price_boundary[call_idx + 1:] = np.nan

    status = np.array(["Short open"] * n, dtype=object)
    status[call_idx:] = "Liquidated"

    return {
        "account_equity": account_equity,
        "equity_before_liquidation": equity_before_liquidation,
        "maintenance_required": maintenance_required,
        "margin_ratio": margin_ratio,
        "call_price_boundary": call_price_boundary,
        "borrow_fees": borrow_fees,
        "unrealized_pnl": unrealized_pnl,
        "status": status,
        "margin_call_idx": call_idx,
        "cover_price": cover_price,
        "liquidation_equity": liquidation_equity,
        "realized_short_pnl": realized_short_pnl,
    }


# ============================================================
# Build data
# ============================================================

market = generate_stock_path()
dates = market["dates"]
prices = market["prices"]
squeeze_start = market["squeeze_start"]

account = simulate_short_account(dates, prices)
account_equity = account["account_equity"]
equity_before_liquidation = account["equity_before_liquidation"]
maintenance_required = account["maintenance_required"]
margin_ratio = account["margin_ratio"]
call_price_boundary = account["call_price_boundary"]
borrow_fees = account["borrow_fees"]
unrealized_pnl = account["unrealized_pnl"]
status = account["status"]
margin_call_idx = account["margin_call_idx"]
cover_price = account["cover_price"]
liquidation_equity = account["liquidation_equity"]
realized_short_pnl = account["realized_short_pnl"]

short_notional = ENTRY_PRICE * SHARES_SHORT
initial_margin_ratio = STARTING_CAPITAL / short_notional
stock_trough_idx = int(np.argmin(prices[:margin_call_idx + 1]))

# Sanity checks make the teaching example fail loudly if assumptions are changed.
if not prices[stock_trough_idx] < ENTRY_PRICE:
    raise RuntimeError("The short should initially become profitable.")
if not account_equity[stock_trough_idx] > STARTING_CAPITAL:
    raise RuntimeError("Equity should rise while the stock falls.")
if not margin_ratio[margin_call_idx] <= MAINTENANCE_MARGIN:
    raise RuntimeError("Margin call must occur at or below maintenance margin.")
if not np.allclose(account_equity[margin_call_idx:], liquidation_equity):
    raise RuntimeError("Equity should be flat after forced liquidation.")


# ============================================================
# Stable plotting ranges
# ============================================================

equity_y_range = padded_range(
    np.concatenate(
        [
            account_equity,
            maintenance_required[np.isfinite(maintenance_required)],
            np.array([STARTING_CAPITAL]),
        ]
    ),
    pad_fraction=0.12,
    min_pad=3_000.0,
)

price_y_range = padded_range(
    np.concatenate(
        [
            prices,
            call_price_boundary[np.isfinite(call_price_boundary)],
            np.array([ENTRY_PRICE]),
        ]
    ),
    pad_fraction=0.10,
    min_pad=5.0,
)


# ============================================================
# Figure layout: simple side-by-side panels
# ============================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.50, 0.50],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Short seller account equity<br><sup>Profits when the stock falls; losses grow when it rises</sup>",
        "Stock price and margin-call boundary<br><sup>Crossing the dashed line forces the broker to buy back shares</sup>",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================

def equity_trace(n_points: int):
    visible_equity = account_equity[:n_points]
    visible_status = status[:n_points]
    visible_pnl = unrealized_pnl[:n_points]
    visible_fees = borrow_fees[:n_points]
    visible_ratio = margin_ratio[:n_points]

    customdata = np.column_stack(
        [visible_pnl, visible_fees, visible_ratio, visible_status]
    )

    return go.Scatter(
        x=dates[:n_points],
        y=visible_equity,
        mode="lines+text",
        line=dict(color=equity_green, width=4),
        text=endpoint_text(f"  Equity {money(visible_equity[-1])}", n_points),
        textposition="middle right",
        textfont=dict(color=equity_green, size=11),
        name="Account equity",
        legendgroup="equity",
        showlegend=True,
        customdata=customdata,
        hovertemplate=(
            "<b>Short account</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Account equity: $%{y:,.0f}<br>"
            "Short P&L before cover: $%{customdata[0]:,.0f}<br>"
            "Borrow fees: $%{customdata[1]:,.0f}<br>"
            "Margin ratio: %{customdata[2]:.1%}<br>"
            "Status: %{customdata[3]}"
            "<extra></extra>"
        ),
    )


def maintenance_trace(n_points: int):
    return go.Scatter(
        x=dates[:n_points],
        y=maintenance_required[:n_points],
        mode="lines",
        line=dict(color=margin_red, width=2.5, dash="dash"),
        name=f"Maintenance requirement ({MAINTENANCE_MARGIN:.0%})",
        legendgroup="maintenance",
        showlegend=True,
        hovertemplate=(
            "<b>Broker maintenance requirement</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Required equity: $%{y:,.0f}"
            "<extra></extra>"
        ),
    )


def equity_margin_call_marker(n_points: int):
    if n_points <= margin_call_idx:
        x = []
        y = []
    else:
        x = [dates[margin_call_idx]]
        y = [account_equity[margin_call_idx]]

    return go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        marker=dict(
            color=margin_red,
            size=13,
            symbol="x",
            line=dict(color=marker_white, width=1.5),
        ),
        text=["  Forced liquidation"] if len(x) else [],
        textposition="bottom right",
        textfont=dict(color=margin_red, size=11),
        name="Forced liquidation",
        legendgroup="event",
        showlegend=False,
        hovertemplate=(
            "<b>Forced liquidation</b><br>"
            f"Broker buys back {SHARES_SHORT:,} shares<br>"
            f"Cover price: ${cover_price:,.2f}<br>"
            f"Equity after cover: ${liquidation_equity:,.0f}"
            "<extra></extra>"
        ),
    )


def stock_price_trace(n_points: int):
    visible_prices = prices[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible_prices,
        mode="lines+text",
        line=dict(color=stock_blue, width=4),
        text=endpoint_text(f"  Stock {money(visible_prices[-1])}", n_points),
        textposition="middle right",
        textfont=dict(color=stock_blue, size=11),
        name="Stock price",
        legendgroup="stock",
        showlegend=True,
        customdata=(visible_prices / ENTRY_PRICE - 1.0),
        hovertemplate=(
            "<b>Stock</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Price: $%{y:,.2f}<br>"
            "Change from short entry: %{customdata:+.1%}"
            "<extra></extra>"
        ),
    )


def call_boundary_trace(n_points: int):
    return go.Scatter(
        x=dates[:n_points],
        y=call_price_boundary[:n_points],
        mode="lines",
        line=dict(color=margin_red, width=2.5, dash="dash"),
        name="Margin-call price boundary",
        legendgroup="call_boundary",
        showlegend=True,
        hovertemplate=(
            "<b>Margin-call boundary</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Boundary price: $%{y:,.2f}<br>"
            f"Maintenance margin: {MAINTENANCE_MARGIN:.0%}"
            "<extra></extra>"
        ),
    )


def short_entry_marker(n_points: int):
    return go.Scatter(
        x=[dates[0]] if n_points >= 1 else [],
        y=[prices[0]] if n_points >= 1 else [],
        mode="markers+text",
        marker=dict(
            color=borrow_orange,
            size=11,
            symbol="triangle-down",
            line=dict(color=marker_white, width=1.0),
        ),
        text=["  Sell borrowed shares"] if n_points >= 1 else [],
        textposition="top right",
        textfont=dict(color=borrow_orange, size=11),
        name="Open short",
        legendgroup="entry",
        showlegend=False,
        hovertemplate=(
            "<b>Open short</b><br>"
            f"Borrow and sell {SHARES_SHORT:,} shares<br>"
            f"Entry price: ${ENTRY_PRICE:,.2f}<br>"
            f"Short-sale proceeds: ${short_notional:,.0f}"
            "<extra></extra>"
        ),
    )


def stock_margin_call_marker(n_points: int):
    if n_points <= margin_call_idx:
        x = []
        y = []
    else:
        x = [dates[margin_call_idx]]
        y = [prices[margin_call_idx]]

    return go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        marker=dict(
            color=margin_red,
            size=13,
            symbol="x",
            line=dict(color=marker_white, width=1.5),
        ),
        text=["  Margin call"] if len(x) else [],
        textposition="top right",
        textfont=dict(color=margin_red, size=11),
        name="Margin call",
        legendgroup="event",
        showlegend=False,
        hovertemplate=(
            "<b>Margin call</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Stock price: $%{y:,.2f}<br>"
            f"Maintenance threshold: {MAINTENANCE_MARGIN:.0%}"
            "<extra></extra>"
        ),
    )


def traces_for_n(n_points: int):
    # The order here must exactly match the initial trace order below.
    return [
        equity_trace(n_points),
        maintenance_trace(n_points),
        equity_margin_call_marker(n_points),
        stock_price_trace(n_points),
        call_boundary_trace(n_points),
        short_entry_marker(n_points),
        stock_margin_call_marker(n_points),
    ]


# ============================================================
# Dynamic title text for each phase
# ============================================================


def full_title(frame_position: int) -> str:
    return (
        "Short Selling Mechanics: Profit, Loss, Margin Call, and Forced Liquidation"
    )


# ============================================================
# Initial traces in the exact order used by every frame
# ============================================================

fig.add_trace(equity_trace(initial_n), row=1, col=1)
fig.add_trace(maintenance_trace(initial_n), row=1, col=1)
fig.add_trace(equity_margin_call_marker(initial_n), row=1, col=1)
fig.add_trace(stock_price_trace(initial_n), row=1, col=2)
fig.add_trace(call_boundary_trace(initial_n), row=1, col=2)
fig.add_trace(short_entry_marker(initial_n), row=1, col=2)
fig.add_trace(stock_margin_call_marker(initial_n), row=1, col=2)


# ============================================================
# Baseline guides
# ============================================================

fig.add_hline(
    y=STARTING_CAPITAL,
    line=dict(color=baseline_color, width=1.4, dash="dot"),
    opacity=0.85,
    row=1,
    col=1,
)

fig.add_hline(
    y=ENTRY_PRICE,
    line=dict(color=baseline_color, width=1.4, dash="dot"),
    opacity=0.85,
    row=1,
    col=2,
)


# ============================================================
# Animation frames and slider
# ============================================================

frames = []
slider_steps = []

frame_positions = list(range(0, len(dates), FRAME_STRIDE))
if frame_positions[-1] != len(dates) - 1:
    frame_positions.append(len(dates) - 1)

for key_idx in [stock_trough_idx, squeeze_start, margin_call_idx]:
    if key_idx not in frame_positions:
        frame_positions.append(key_idx)

frame_positions = sorted(set(frame_positions))

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_name = f"frame_{frame_position}"
    frame_data = traces_for_n(n_points)

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
            layout=go.Layout(title=dict(text=full_title(frame_position))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %d"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================

for col in [1, 2]:
    fig.update_xaxes(
        axis_style,
        row=1,
        col=col,
        range=[dates[0], dates[-1]],
        title_text="Trading date",
        title_standoff=20,
    )

fig.update_yaxes(
    axis_style,
    row=1,
    col=1,
    range=equity_y_range,
    title_text="Account equity / required equity",
    tickprefix="$",
    separatethousands=True,
)

fig.update_yaxes(
    axis_style,
    row=1,
    col=2,
    range=price_y_range,
    title_text="Stock price",
    tickprefix="$",
    separatethousands=True,
)

fig.update_layout(
    title=dict(
        text=full_title(0),
        x=0.5,
        font=dict(color=off_white),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=FIG_HEIGHT,
    width=FIG_WIDTH,
    margin=dict(t=140, b=220, r=80, l=90),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.16,
        yanchor="top",
        font=dict(color=off_white),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.31,
            "yanchor": "top",
            "pad": {"r": 10, "t": 38},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": FRAME_DURATION,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.31,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Date: ",
                "font": {"color": muted_white},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=off_white, size=14))

# Compact formula note below the charts but above the controls.
fig.add_annotation(
    x=0.5,
    y=-0.10,
    xref="paper",
    yref="paper",
    text=(
        "Short P&amp;L = (entry price − current price) × shares"
        " &nbsp; | &nbsp; "
        "Margin ratio = account equity ÷ current short market value"
    ),
    showarrow=False,
    font=dict(color=muted_white, size=11),
    align="center",
)

# Static event summary placed unobtrusively in the right panel.
fig.add_annotation(
    x=dates[margin_call_idx],
    y=price_y_range[1] - 0.06 * (price_y_range[1] - price_y_range[0]),
    xref="x2",
    yref="y2",
    text=(
        f"Call at {dates[margin_call_idx].strftime('%b %d')}<br>"
        f"Stock ${prices[margin_call_idx]:,.2f} | "
        f"Equity {money(liquidation_equity)}"
    ),
    showarrow=False,
    font=dict(color=off_white, size=10),
    bgcolor="rgba(0,0,0,0.45)",
    bordercolor="rgba(255,255,255,0.18)",
    borderwidth=1,
    borderpad=5,
)



###### ______________________________________________________________________________________________________________________________________

##### 📈 Long-short trading strategies

 Long-short strategies involve taking long positions in assets expected to rise and short positions in assets expected to fall.
 
  In a long-short portfolio, total exposure can exceed initial capital by using leverage. For example:
  
  $$
  \text{Long:}\ \$100{,}000,\quad \text{Short:}\ -\$100{,}000 \implies \text{Gross Exposure} = \$200{,}000,\quad \text{Leverage} = 2\times
  $$
  
  Here, capital is fully used to support both long and short positions.

In [9]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# Configuration
# ============================================================


@dataclass(frozen=True)
class Config:
    # Animation / output
    random_seed: int = 21
    frame_duration_ms: int = 85
    frame_stride: int = 2
    figure_width: int = 1_200
    figure_height: int = 720
    output_html: Path = Path("/mnt/data/long_short_margin_call_animation.html")
    write_html: bool = True
    show_figure: bool = False

    # Portfolio construction
    starting_equity: float = 70_000.0
    long_entry_price: float = 100.0
    short_entry_price: float = 100.0
    long_shares: int = 1_000
    short_shares: int = 1_000

    # Simplified broker maintenance rules
    long_maintenance_margin: float = 0.25
    short_maintenance_margin: float = 0.30
    annual_short_borrow_fee: float = 0.04

    # Forced-liquidation costs
    long_sell_slippage_bps: float = 10.0
    short_cover_slippage_bps: float = 15.0
    commission_per_leg: float = 5.0


CFG = Config()


# ============================================================
# Styling
# ============================================================

OFF_WHITE = "#e0e0e0"
MUTED_WHITE = "#aaaaaa"
AXIS_GRID = "rgba(255,255,255,0.10)"
BASELINE = "#777777"

LONG_BLUE = "#4da3ff"
SHORT_ORANGE = "#ff9f43"
TOTAL_GREEN = "#29d17d"
MARGIN_RED = "#ff4d5a"
MARKER_WHITE = "#f4f4f4"

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor=AXIS_GRID,
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
)


# ============================================================
# Small helpers
# ============================================================


def padded_range(values, pad_fraction: float = 0.10, min_pad: float = 1.0) -> list[float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return [-1.0, 1.0]

    low = float(values.min())
    high = float(values.max())
    if np.isclose(low, high):
        pad = max(abs(high) * pad_fraction, min_pad)
    else:
        pad = max((high - low) * pad_fraction, min_pad)
    return [low - pad, high + pad]


def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


def money(value: float) -> str:
    return f"${value:,.0f}"


def pct(value: float) -> str:
    return f"{100.0 * value:.1f}%"


# ============================================================
# Generate a deterministic pair-trade story
#
# The long thesis works throughout. The short thesis works first,
# then the short stock gaps higher and squeezes. The squeeze becomes
# large enough that total account equity falls below the broker's
# maintenance requirement on the combined gross positions.
# ============================================================


def generate_price_paths(seed: int = CFG.random_seed) -> dict[str, object]:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2027-01-04", periods=140)
    n = len(dates)

    long_returns = np.zeros(n, dtype=float)
    short_returns = np.zeros(n, dtype=float)

    # Long leg: a fairly steady winner.
    long_returns[1:35] = rng.normal(0.0010, 0.0045, 34)
    long_returns[35:70] = rng.normal(0.0012, 0.0040, 35)
    long_returns[70:110] = rng.normal(0.0010, 0.0050, 40)
    long_returns[110:] = rng.normal(0.0005, 0.0045, n - 110)

    # Short leg: initially behaves as expected, then experiences a squeeze.
    short_returns[1:30] = rng.normal(-0.0012, 0.0045, 29)
    short_returns[30:55] = rng.normal(-0.0030, 0.0055, 25)
    short_returns[55] = 0.060  # gap higher starts the squeeze
    short_returns[56:100] = rng.normal(0.0090, 0.0090, 44)
    short_returns[100:] = rng.normal(0.0015, 0.0060, n - 100)

    long_prices = np.empty(n, dtype=float)
    short_prices = np.empty(n, dtype=float)
    long_prices[0] = CFG.long_entry_price
    short_prices[0] = CFG.short_entry_price

    for i in range(1, n):
        long_prices[i] = long_prices[i - 1] * (1.0 + long_returns[i])
        short_prices[i] = short_prices[i - 1] * (1.0 + short_returns[i])

    return {
        "dates": dates,
        "long_returns": long_returns,
        "short_returns": short_returns,
        "long_prices": long_prices,
        "short_prices": short_prices,
        "squeeze_start": 55,
    }


# ============================================================
# Simulate the long/short account
#
# Opening trade:
#   buy the long shares
#   borrow and sell the short shares
#
# Account equity while positions are open:
#   cash after opening + long market value - short market value - borrow fees
#
# Simplified maintenance requirement:
#   25% of long market value + 30% of short market value
#
# The short-price boundary solves for the price at which:
#   account equity == maintenance requirement
# while holding the current long price and accrued borrow fees fixed.
# ============================================================


def simulate_account(dates, long_prices, short_prices) -> dict[str, object]:
    n = len(dates)

    long_entry_notional = CFG.long_entry_price * CFG.long_shares
    short_entry_notional = CFG.short_entry_price * CFG.short_shares
    cash_after_open = CFG.starting_equity - long_entry_notional + short_entry_notional
    daily_borrow_rate = CFG.annual_short_borrow_fee / 252.0

    borrow_fees = np.zeros(n, dtype=float)
    long_pnl = np.zeros(n, dtype=float)
    short_pnl = np.zeros(n, dtype=float)
    total_equity_before_liquidation = np.zeros(n, dtype=float)
    long_sleeve_equity = np.zeros(n, dtype=float)
    short_sleeve_equity = np.zeros(n, dtype=float)
    maintenance_required = np.full(n, np.nan, dtype=float)
    margin_cushion = np.full(n, np.nan, dtype=float)
    short_call_boundary = np.full(n, np.nan, dtype=float)

    sleeve_start = CFG.starting_equity / 2.0
    margin_call_idx = None

    for i in range(n):
        if i > 0:
            # Borrow fee is charged on the previous day's short market value.
            borrow_fees[i] = (
                borrow_fees[i - 1]
                + daily_borrow_rate * CFG.short_shares * short_prices[i - 1]
            )

        long_market_value = CFG.long_shares * long_prices[i]
        short_market_value = CFG.short_shares * short_prices[i]

        long_pnl[i] = long_market_value - long_entry_notional
        short_pnl[i] = short_entry_notional - short_market_value

        long_sleeve_equity[i] = sleeve_start + long_pnl[i]
        short_sleeve_equity[i] = sleeve_start + short_pnl[i] - borrow_fees[i]

        total_equity_before_liquidation[i] = (
            cash_after_open
            + long_market_value
            - short_market_value
            - borrow_fees[i]
        )

        maintenance_required[i] = (
            CFG.long_maintenance_margin * long_market_value
            + CFG.short_maintenance_margin * short_market_value
        )
        margin_cushion[i] = (
            total_equity_before_liquidation[i] - maintenance_required[i]
        )

        # Solve the maintenance equation for the short stock price.
        short_call_boundary[i] = (
            cash_after_open
            + (1.0 - CFG.long_maintenance_margin) * long_market_value
            - borrow_fees[i]
        ) / ((1.0 + CFG.short_maintenance_margin) * CFG.short_shares)

        if (
            margin_call_idx is None
            and total_equity_before_liquidation[i] <= maintenance_required[i]
        ):
            margin_call_idx = i
            break

    if margin_call_idx is None:
        raise RuntimeError(
            "The generated scenario did not trigger a margin call. "
            "Increase the short squeeze or reduce starting equity."
        )

    call_idx = margin_call_idx

    # Freeze pre-liquidation accounting arrays after the call day.
    for array in (
        borrow_fees,
        long_pnl,
        short_pnl,
        total_equity_before_liquidation,
        long_sleeve_equity,
        short_sleeve_equity,
        margin_cushion,
    ):
        array[call_idx + 1 :] = array[call_idx]

    # Execute both legs adversely when the broker liquidates the account.
    long_exit_price = long_prices[call_idx] * (
        1.0 - CFG.long_sell_slippage_bps / 10_000.0
    )
    short_cover_price = short_prices[call_idx] * (
        1.0 + CFG.short_cover_slippage_bps / 10_000.0
    )

    realized_long_pnl = (
        (long_exit_price - CFG.long_entry_price) * CFG.long_shares
        - CFG.commission_per_leg
    )
    realized_short_pnl = (
        (CFG.short_entry_price - short_cover_price) * CFG.short_shares
        - borrow_fees[call_idx]
        - CFG.commission_per_leg
    )

    final_long_sleeve_equity = sleeve_start + realized_long_pnl
    final_short_sleeve_equity = sleeve_start + realized_short_pnl
    liquidation_equity = final_long_sleeve_equity + final_short_sleeve_equity

    total_equity = total_equity_before_liquidation.copy()
    total_equity[call_idx:] = liquidation_equity
    long_sleeve_equity[call_idx:] = final_long_sleeve_equity
    short_sleeve_equity[call_idx:] = final_short_sleeve_equity

    # The broker requirement and price boundary disappear once both positions close.
    maintenance_required[call_idx + 1 :] = np.nan
    short_call_boundary[call_idx + 1 :] = np.nan

    return {
        "cash_after_open": cash_after_open,
        "borrow_fees": borrow_fees,
        "long_pnl": long_pnl,
        "short_pnl": short_pnl,
        "total_equity": total_equity,
        "total_equity_before_liquidation": total_equity_before_liquidation,
        "long_sleeve_equity": long_sleeve_equity,
        "short_sleeve_equity": short_sleeve_equity,
        "maintenance_required": maintenance_required,
        "margin_cushion": margin_cushion,
        "short_call_boundary": short_call_boundary,
        "margin_call_idx": call_idx,
        "long_exit_price": long_exit_price,
        "short_cover_price": short_cover_price,
        "realized_long_pnl": realized_long_pnl,
        "realized_short_pnl": realized_short_pnl,
        "liquidation_equity": liquidation_equity,
    }


# ============================================================
# Build and validate the teaching scenario
# ============================================================


market = generate_price_paths()
dates = market["dates"]
long_prices = market["long_prices"]
short_prices = market["short_prices"]
squeeze_start = int(market["squeeze_start"])

account = simulate_account(dates, long_prices, short_prices)
borrow_fees = account["borrow_fees"]
long_pnl = account["long_pnl"]
short_pnl = account["short_pnl"]
total_equity = account["total_equity"]
total_equity_before_liquidation = account["total_equity_before_liquidation"]
long_sleeve_equity = account["long_sleeve_equity"]
short_sleeve_equity = account["short_sleeve_equity"]
maintenance_required = account["maintenance_required"]
margin_cushion = account["margin_cushion"]
short_call_boundary = account["short_call_boundary"]
margin_call_idx = int(account["margin_call_idx"])
long_exit_price = float(account["long_exit_price"])
short_cover_price = float(account["short_cover_price"])
realized_long_pnl = float(account["realized_long_pnl"])
realized_short_pnl = float(account["realized_short_pnl"])
liquidation_equity = float(account["liquidation_equity"])

long_entry_notional = CFG.long_entry_price * CFG.long_shares
short_entry_notional = CFG.short_entry_price * CFG.short_shares
initial_gross_exposure = long_entry_notional + short_entry_notional
initial_net_exposure = long_entry_notional - short_entry_notional
short_trough_idx = int(np.argmin(short_prices[: margin_call_idx + 1]))

# Fail loudly if edits break the intended message.
if not long_sleeve_equity[margin_call_idx] > CFG.starting_equity / 2.0:
    raise RuntimeError("The long leg must still be profitable at the margin call.")
if not short_sleeve_equity[margin_call_idx] < CFG.starting_equity / 2.0:
    raise RuntimeError("The short leg must be losing money at the margin call.")
if not total_equity_before_liquidation[margin_call_idx] <= maintenance_required[margin_call_idx]:
    raise RuntimeError("The call must occur when equity reaches maintenance.")
if not np.allclose(total_equity[margin_call_idx:], liquidation_equity):
    raise RuntimeError("Total equity must be flat after forced liquidation.")
if not short_prices[short_trough_idx] < CFG.short_entry_price:
    raise RuntimeError("The short thesis should initially work before the squeeze.")


# ============================================================
# Stable plot ranges
# ============================================================


equity_y_range = padded_range(
    np.concatenate(
        [
            total_equity,
            long_sleeve_equity,
            short_sleeve_equity,
            maintenance_required[np.isfinite(maintenance_required)],
            np.array([CFG.starting_equity, CFG.starting_equity / 2.0]),
        ]
    ),
    pad_fraction=0.12,
    min_pad=4_000.0,
)

price_y_range = padded_range(
    np.concatenate(
        [
            long_prices,
            short_prices,
            short_call_boundary[np.isfinite(short_call_boundary)],
            np.array([CFG.long_entry_price, CFG.short_entry_price]),
        ]
    ),
    pad_fraction=0.10,
    min_pad=5.0,
)


# ============================================================
# Figure: two side-by-side panels
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.53, 0.47],
    horizontal_spacing=0.10,
    subplot_titles=(
        "Strategy equity by sleeve<br><sup>The long leg can win while the short leg forces liquidation</sup>",
        "Long and short price paths<br><sup>The short stock crossing its dashed boundary triggers the call</sup>",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================


def total_equity_trace(n_points: int) -> go.Scatter:
    visible = total_equity[:n_points]
    customdata = np.column_stack(
        [
            long_pnl[:n_points],
            short_pnl[:n_points],
            borrow_fees[:n_points],
            maintenance_required[:n_points],
            margin_cushion[:n_points],
        ]
    )
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=TOTAL_GREEN, width=4.5),
        text=endpoint_text(f"  Total {money(visible[-1])}", n_points),
        textposition="middle right",
        textfont=dict(color=TOTAL_GREEN, size=11),
        name="Total strategy equity",
        legendgroup="total_equity",
        showlegend=True,
        customdata=customdata,
        hovertemplate=(
            "<b>Total strategy</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Account equity: $%{y:,.0f}<br>"
            "Long P&L: $%{customdata[0]:,.0f}<br>"
            "Short P&L: $%{customdata[1]:,.0f}<br>"
            "Borrow fees: $%{customdata[2]:,.0f}<br>"
            "Maintenance required: $%{customdata[3]:,.0f}<br>"
            "Margin cushion: $%{customdata[4]:,.0f}"
            "<extra></extra>"
        ),
    )


def sleeve_trace(
    label: str,
    values,
    pnl_values,
    color: str,
    text_position: str,
    n_points: int,
) -> go.Scatter:
    visible = values[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=color, width=3.2),
        text=endpoint_text(f"  {label} {money(visible[-1])}", n_points),
        textposition=text_position,
        textfont=dict(color=color, size=10),
        name=f"{label} sleeve equity",
        legendgroup=f"{label.lower()}_sleeve",
        showlegend=True,
        customdata=pnl_values[:n_points],
        hovertemplate=(
            f"<b>{label} sleeve</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Attributed sleeve equity: $%{y:,.0f}<br>"
            "Leg P&L: $%{customdata:,.0f}"
            "<extra></extra>"
        ),
    )


def maintenance_trace(n_points: int) -> go.Scatter:
    return go.Scatter(
        x=dates[:n_points],
        y=maintenance_required[:n_points],
        mode="lines",
        line=dict(color=MARGIN_RED, width=2.5, dash="dash"),
        name="Broker maintenance requirement",
        legendgroup="maintenance",
        showlegend=True,
        hovertemplate=(
            "<b>Broker maintenance requirement</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Required equity: $%{y:,.0f}<br>"
            f"Long: {CFG.long_maintenance_margin:.0%} of market value<br>"
            f"Short: {CFG.short_maintenance_margin:.0%} of market value"
            "<extra></extra>"
        ),
    )


def equity_call_marker(n_points: int) -> go.Scatter:
    event_visible = n_points > margin_call_idx
    return go.Scatter(
        x=[dates[margin_call_idx]] if event_visible else [],
        y=[liquidation_equity] if event_visible else [],
        mode="markers+text",
        marker=dict(
            color=MARGIN_RED,
            size=14,
            symbol="x",
            line=dict(color=MARKER_WHITE, width=1.5),
        ),
        text=["  Margin call: close both legs"] if event_visible else [],
        textposition="bottom right",
        textfont=dict(color=MARGIN_RED, size=11),
        name="Forced liquidation",
        legendgroup="event",
        showlegend=False,
        hovertemplate=(
            "<b>Forced liquidation</b><br>"
            f"Date: {dates[margin_call_idx].strftime('%Y-%m-%d')}<br>"
            f"Equity before execution: ${total_equity_before_liquidation[margin_call_idx]:,.0f}<br>"
            f"Maintenance required: ${maintenance_required[margin_call_idx]:,.0f}<br>"
            f"Equity after slippage/costs: ${liquidation_equity:,.0f}<br>"
            f"Long realized P&L: ${realized_long_pnl:,.0f}<br>"
            f"Short realized P&L: ${realized_short_pnl:,.0f}"
            "<extra></extra>"
        ),
    )


def price_trace(
    label: str,
    prices,
    entry_price: float,
    color: str,
    text_position: str,
    n_points: int,
) -> go.Scatter:
    visible = prices[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=color, width=4),
        text=endpoint_text(f"  {label} {money(visible[-1])}", n_points),
        textposition=text_position,
        textfont=dict(color=color, size=11),
        name=f"{label} stock",
        legendgroup=f"{label.lower()}_price",
        showlegend=True,
        customdata=visible / entry_price - 1.0,
        hovertemplate=(
            f"<b>{label} stock</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Price: $%{y:,.2f}<br>"
            "Change from entry: %{customdata:+.1%}"
            "<extra></extra>"
        ),
    )


def call_boundary_trace(n_points: int) -> go.Scatter:
    return go.Scatter(
        x=dates[:n_points],
        y=short_call_boundary[:n_points],
        mode="lines",
        line=dict(color=MARGIN_RED, width=2.5, dash="dash"),
        name="Short-leg call boundary",
        legendgroup="call_boundary",
        showlegend=True,
        hovertemplate=(
            "<b>Short-stock call boundary</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Boundary price: $%{y:,.2f}<br>"
            "Above this level, account equity is insufficient"
            "<extra></extra>"
        ),
    )


def entry_markers(n_points: int) -> go.Scatter:
    return go.Scatter(
        x=[dates[0], dates[0]] if n_points >= 1 else [],
        y=[long_prices[0], short_prices[0]] if n_points >= 1 else [],
        mode="markers+text",
        marker=dict(
            color=[LONG_BLUE, SHORT_ORANGE],
            size=11,
            symbol=["triangle-up", "triangle-down"],
            line=dict(color=MARKER_WHITE, width=1.0),
        ),
        text=["  Buy long", "  Sell short"] if n_points >= 1 else [],
        textposition=["top left", "bottom left"] if n_points >= 1 else [],
        textfont=dict(color=OFF_WHITE, size=10),
        name="Open pair trade",
        legendgroup="entry",
        showlegend=False,
        hovertemplate=(
            "<b>Open pair trade</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Price: $%{y:,.2f}"
            "<extra></extra>"
        ),
    )


def price_call_marker(n_points: int) -> go.Scatter:
    event_visible = n_points > margin_call_idx
    return go.Scatter(
        x=[dates[margin_call_idx]] if event_visible else [],
        y=[short_prices[margin_call_idx]] if event_visible else [],
        mode="markers+text",
        marker=dict(
            color=MARGIN_RED,
            size=14,
            symbol="x",
            line=dict(color=MARKER_WHITE, width=1.5),
        ),
        text=["  Short crosses boundary"] if event_visible else [],
        textposition="top right",
        textfont=dict(color=MARGIN_RED, size=11),
        name="Margin call",
        legendgroup="event",
        showlegend=False,
        hovertemplate=(
            "<b>Margin call</b><br>"
            f"Date: {dates[margin_call_idx].strftime('%Y-%m-%d')}<br>"
            f"Short stock: ${short_prices[margin_call_idx]:,.2f}<br>"
            f"Call boundary: ${short_call_boundary[margin_call_idx]:,.2f}<br>"
            f"Long stock: ${long_prices[margin_call_idx]:,.2f}"
            "<extra></extra>"
        ),
    )


def traces_for_n(n_points: int) -> list[go.Scatter]:
    # This order must exactly match the initial trace order below.
    return [
        total_equity_trace(n_points),
        sleeve_trace(
            "Long",
            long_sleeve_equity,
            long_pnl,
            LONG_BLUE,
            "top right",
            n_points,
        ),
        sleeve_trace(
            "Short",
            short_sleeve_equity,
            short_pnl - borrow_fees,
            SHORT_ORANGE,
            "bottom right",
            n_points,
        ),
        maintenance_trace(n_points),
        equity_call_marker(n_points),
        price_trace(
            "Long",
            long_prices,
            CFG.long_entry_price,
            LONG_BLUE,
            "top right",
            n_points,
        ),
        price_trace(
            "Short",
            short_prices,
            CFG.short_entry_price,
            SHORT_ORANGE,
            "bottom right",
            n_points,
        ),
        call_boundary_trace(n_points),
        entry_markers(n_points),
        price_call_marker(n_points),
    ]


# ============================================================
# Dynamic narrative
# ============================================================


def full_title(frame_position: int) -> str:
    return (
        "Long/Short Is Not Margin-Neutral: One Short Squeeze Can Liquidate the Whole Book"
    )


# ============================================================
# Initial traces
# ============================================================


fig.add_trace(total_equity_trace(initial_n), row=1, col=1)
fig.add_trace(
    sleeve_trace(
        "Long", long_sleeve_equity, long_pnl, LONG_BLUE, "top right", initial_n
    ),
    row=1,
    col=1,
)
fig.add_trace(
    sleeve_trace(
        "Short",
        short_sleeve_equity,
        short_pnl - borrow_fees,
        SHORT_ORANGE,
        "bottom right",
        initial_n,
    ),
    row=1,
    col=1,
)
fig.add_trace(maintenance_trace(initial_n), row=1, col=1)
fig.add_trace(equity_call_marker(initial_n), row=1, col=1)

fig.add_trace(
    price_trace(
        "Long",
        long_prices,
        CFG.long_entry_price,
        LONG_BLUE,
        "top right",
        initial_n,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    price_trace(
        "Short",
        short_prices,
        CFG.short_entry_price,
        SHORT_ORANGE,
        "bottom right",
        initial_n,
    ),
    row=1,
    col=2,
)
fig.add_trace(call_boundary_trace(initial_n), row=1, col=2)
fig.add_trace(entry_markers(initial_n), row=1, col=2)
fig.add_trace(price_call_marker(initial_n), row=1, col=2)


# ============================================================
# Baseline guides
# ============================================================


fig.add_hline(
    y=CFG.starting_equity,
    line=dict(color=BASELINE, width=1.4, dash="dot"),
    opacity=0.85,
    row=1,
    col=1,
)
fig.add_hline(
    y=CFG.starting_equity / 2.0,
    line=dict(color=BASELINE, width=1.1, dash="dot"),
    opacity=0.55,
    row=1,
    col=1,
)
fig.add_hline(
    y=100.0,
    line=dict(color=BASELINE, width=1.4, dash="dot"),
    opacity=0.85,
    row=1,
    col=2,
)


# ============================================================
# Animation frames and slider
# ============================================================


frames: list[go.Frame] = []
slider_steps: list[dict] = []

frame_positions = list(range(0, len(dates), CFG.frame_stride))
if frame_positions[-1] != len(dates) - 1:
    frame_positions.append(len(dates) - 1)

for key_idx in [short_trough_idx, squeeze_start, margin_call_idx]:
    if key_idx not in frame_positions:
        frame_positions.append(key_idx)

frame_positions = sorted(set(frame_positions))

for frame_position in frame_positions:
    n_points = frame_position + 1
    frame_name = f"frame_{frame_position}"
    frame_data = traces_for_n(n_points)

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
            layout=go.Layout(title=dict(text=full_title(frame_position))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %d"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


for col in [1, 2]:
    fig.update_xaxes(
        AXIS_STYLE,
        row=1,
        col=col,
        range=[dates[0], dates[-1]],
        title_text="Trading date",
        title_standoff=20,
    )

fig.update_yaxes(
    AXIS_STYLE,
    row=1,
    col=1,
    range=equity_y_range,
    title_text="Equity / maintenance requirement",
    tickprefix="$",
    separatethousands=True,
)

fig.update_yaxes(
    AXIS_STYLE,
    row=1,
    col=2,
    range=price_y_range,
    title_text="Stock price",
    tickprefix="$",
    separatethousands=True,
)

fig.update_layout(
    title=dict(
        text=full_title(0),
        x=0.5,
        font=dict(color=OFF_WHITE),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=CFG.figure_height,
    width=CFG.figure_width,
    margin=dict(t=145, b=235, r=95, l=90),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.17,
        yanchor="top",
        font=dict(color=OFF_WHITE),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.32,
            "yanchor": "top",
            "pad": {"r": 10, "t": 38},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": CFG.frame_duration_ms,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.32,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Date: ",
                "font": {"color": MUTED_WHITE},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=OFF_WHITE, size=14))

# Compact mechanics note between the chart and controls.
fig.add_annotation(
    x=0.5,
    y=-0.105,
    xref="paper",
    yref="paper",
    text=(
        "Equity = cash + long market value − short market value − borrow fees"
        " &nbsp; | &nbsp; "
        f"Maintenance = {CFG.long_maintenance_margin:.0%} × long MV + "
        f"{CFG.short_maintenance_margin:.0%} × short MV"
    ),
    showarrow=False,
    font=dict(color=MUTED_WHITE, size=11),
    align="center",
)

# A concise summary near the call point. It is intentionally placed inside
# the equity panel so the central lesson can be read without hovering.
fig.add_annotation(
    x=dates[margin_call_idx],
    y=equity_y_range[1] - 0.06 * (equity_y_range[1] - equity_y_range[0]),
    xref="x",
    yref="y",
    text=(
        f"Call on {dates[margin_call_idx].strftime('%b %d')}<br>"
        f"Long P&amp;L {money(realized_long_pnl)} | "
        f"Short P&amp;L {money(realized_short_pnl)}<br>"
        f"Final equity {money(liquidation_equity)}"
    ),
    showarrow=False,
    font=dict(color=OFF_WHITE, size=10),
    bgcolor="rgba(0,0,0,0.48)",
    bordercolor="rgba(255,255,255,0.18)",
    borderwidth=1,
    borderpad=5,
)


###### ______________________________________________________________________________________________________________________________________

##### ⛵ Why this didn't have to happen

Situational Awareness LP didn't have to lose $35 billion—basic risk management would have changed everything.

 They could have hedged their exposures, either statically (using options, futures, etc.) or dynamically, so that even a large market move wouldn't have led to catastrophic losses.

 They also could have simply run a single stress test: take last year's positions, apply a range of adverse scenarios (like a sudden 10% move, spike in volatility, liquidity freeze, etc.) and immediately see how vulnerable they were. Any half-decent stress test would have exposed the risk concentration and potential losses.

 Ultimately, every large financial blowup comes down to failing to imagine a possible scenario—and refusing to check your models under real-world stress. The models are always simplifications, and it's human choices (complacency, hubris, oversight) that decide how they're used. 

 The real world keeps changing, so every model and set of parameters needs to be questioned and updated regularly. If you don't, you end up with a Black Swan that was perfectly preventable—just another stress test away.

In [10]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# ============================================================
# PURPOSE AND SOURCE BOUNDARIES
# ============================================================
#
# This is an ILLUSTRATIVE reconstruction, not Situational Awareness LP's
# internal risk report. Public sources do not disclose the fund's exact NAV,
# financing terms, option strikes/expiries, deltas, short-equity positions,
# VaR model, limits, or prime-broker margin schedule.
#
# Public anchors used in the story:
#   1) Q1 2026 Form 13F (period ended 2026-03-31):
#        common-stock value                 $3.856B
#        call-option underlying value       $1.362B
#        put-option underlying value        $8.459B
#        total reported 13F value          $13.677B
#      SEC filing:
#      https://www.sec.gov/Archives/edgar/data/2045724/000204572426000008/
#
#      Important: 13F options are reported in terms of the underlying
#      securities. The reported values are not option premiums or exact
#      delta-adjusted exposures. Form 13F also excludes short-equity positions.
#
#   2) Reuters reported that the fund:
#        gained 439% from the start of 2026 through June,
#        lost 67% in July,
#        unwound most of a roughly $16B public-equities portfolio,
#        sold the bulk of that portfolio to Citadel,
#        and removed all leverage after the episode.
#
# The risk metrics and stress test below are transparent model assumptions.
# They are designed to show how a leveraged, basis-sensitive long/short book
# can retain enormous tail risk even after exceptional gains.
# ============================================================


@dataclass(frozen=True)
class Config:
    # Animation and output
    random_seed: int = 47
    frame_duration_ms: int = 75
    frame_stride: int = 2
    figure_width: int = 1_260
    figure_height: int = 760
    output_html: Path = Path(
        "/mnt/data/situational_awareness_var_stress_animation_v2.html"
    )
    write_html: bool = True
    show_figure: bool = False

    # Reported performance anchors, expressed as a NAV index
    starting_nav: float = 100.0
    reported_return_through_june: float = 4.39
    reported_july_drawdown: float = -0.67

    # Public 13F snapshot, dollars
    common_stock_value: float = 3_855_771_552.0
    call_underlying_value: float = 1_361_829_026.0
    put_underlying_value: float = 8_459_056_999.0

    # Illustrative delta assumptions for the option lines.
    # These are not known fund deltas.
    assumed_call_delta: float = 0.55
    assumed_put_delta: float = -0.45

    # Illustrative gross exposure after scaling risk with gains.
    # At the June peak, the model runs 2.5x long and 1.5x short = 4.0x gross.
    start_long_multiple: float = 1.30
    start_short_multiple: float = 0.90
    peak_long_multiple: float = 2.50
    peak_short_multiple: float = 1.50

    # One-week factor-risk assumptions
    long_daily_vol: float = 0.035
    short_basket_daily_vol: float = 0.030
    normal_factor_correlation: float = 0.55
    student_t_degrees_freedom: int = 5
    var_confidence: float = 0.99
    monte_carlo_scenarios: int = 120_000

    # Simple bad-week stress test at peak leverage
    stress_long_return: float = -0.20
    stress_short_basket_return: float = 0.07
    stress_liquidity_and_financing: float = -0.02

    # Illustrative governance thresholds, not reported fund terms
    one_week_risk_limit: float = 0.25
    intervention_drawdown_from_peak: float = 0.50


CFG = Config()


# ============================================================
# Styling
# ============================================================

OFF_WHITE = "#e0e0e0"
MUTED_WHITE = "#aaaaaa"
AXIS_GRID = "rgba(255,255,255,0.10)"
BASELINE = "#777777"

TOTAL_GREEN = "#29d17d"
LONG_BLUE = "#4da3ff"
SHORT_ORANGE = "#ff9f43"
VAR_YELLOW = "#ffd166"
CVAR_ORANGE = "#ff9f43"
STRESS_RED = "#ff4d5a"
REPORTED_PURPLE = "#b28dff"
MARKER_WHITE = "#f4f4f4"

AXIS_STYLE = dict(
    showgrid=True,
    gridcolor=AXIS_GRID,
    tickfont=dict(color=OFF_WHITE),
    linecolor=OFF_WHITE,
    zeroline=False,
    title_font=dict(color=OFF_WHITE),
)


# ============================================================
# Helpers
# ============================================================


def endpoint_text(label: str, n_points: int) -> list[str]:
    return [""] * max(0, n_points - 1) + [label]


def pct(value: float, decimals: int = 1) -> str:
    return f"{100.0 * value:.{decimals}f}%"


def billions(value: float) -> str:
    return f"${value / 1e9:,.2f}B"


def smooth_bridge(
    n_points: int,
    start_value: float,
    end_value: float,
    rng: np.random.Generator,
    noise_scale: float,
) -> np.ndarray:
    """Create a smooth positive path pinned exactly to start and end."""
    if n_points < 2:
        return np.array([end_value], dtype=float)

    t = np.linspace(0.0, 1.0, n_points)
    raw_noise = rng.normal(0.0, 1.0, n_points)
    smoothed_noise = pd.Series(raw_noise).rolling(9, center=True, min_periods=1).mean().to_numpy()

    # Brownian-bridge style noise is exactly zero at both endpoints.
    bridge = smoothed_noise - ((1.0 - t) * smoothed_noise[0] + t * smoothed_noise[-1])
    bridge *= np.sin(np.pi * t)

    log_start = np.log(start_value)
    log_end = np.log(end_value)
    log_path = (1.0 - t) * log_start + t * log_end + noise_scale * bridge
    path = np.exp(log_path)
    path[0] = start_value
    path[-1] = end_value
    return path


# ============================================================
# Build an attributed NAV history pinned to the reported endpoints
# ============================================================


def build_nav_history(seed: int = CFG.random_seed) -> dict[str, object]:
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2026-01-02", "2026-07-31")

    june_end = dates.get_loc(pd.Timestamp("2026-06-30"))
    stress_start = dates.get_loc(pd.Timestamp("2026-07-27"))
    final_idx = len(dates) - 1

    june_peak_nav = CFG.starting_nav * (1.0 + CFG.reported_return_through_june)
    july_end_nav = june_peak_nav * (1.0 + CFG.reported_july_drawdown)

    # The public reporting says the drawdown was concentrated in July and was
    # exacerbated by extreme moves in core positions during the final week.
    pre_stress_nav = june_peak_nav * 0.81

    # Attribute the strategy NAV into two capital sleeves. These are teaching
    # attributions, not reported sleeve accounts.
    long_start = CFG.starting_nav * 0.50
    short_start = CFG.starting_nav * 0.50

    long_june = june_peak_nav * 0.70
    short_june = june_peak_nav - long_june

    long_pre_stress = pre_stress_nav * 0.75
    short_pre_stress = pre_stress_nav - long_pre_stress

    long_final = july_end_nav * 0.84
    short_final = july_end_nav - long_final

    n_pre_june = june_end + 1
    n_early_july = stress_start - june_end - 1
    n_stress = final_idx - stress_start + 1

    long_pre_june = smooth_bridge(
        n_pre_june, long_start, long_june, rng, noise_scale=0.050
    )
    short_pre_june = smooth_bridge(
        n_pre_june, short_start, short_june, rng, noise_scale=0.045
    )

    # Early July erosion before the final five-day break.
    long_early_july = smooth_bridge(
        n_early_july + 1,
        long_june,
        long_pre_stress,
        rng,
        noise_scale=0.018,
    )[1:]
    short_early_july = smooth_bridge(
        n_early_july + 1,
        short_june,
        short_pre_stress,
        rng,
        noise_scale=0.018,
    )[1:]

    # Final bad week: both sleeves lose money. The long high-beta sleeve falls,
    # while the short/put hedge also loses because the hedge basket rallies and
    # the relative-value relationship breaks.
    stress_progress = np.array([0.00, 0.12, 0.31, 0.56, 0.79, 1.00])
    if n_stress + 1 != len(stress_progress):
        raise RuntimeError("Expected five business days in the stress week.")

    long_stress = long_pre_stress * np.exp(
        np.log(long_final / long_pre_stress) * stress_progress
    )
    short_stress = short_pre_stress * np.exp(
        np.log(short_final / short_pre_stress) * stress_progress
    )

    long_sleeve = np.concatenate(
        [long_pre_june, long_early_july, long_stress[1:]]
    )
    short_sleeve = np.concatenate(
        [short_pre_june, short_early_july, short_stress[1:]]
    )
    total_nav = long_sleeve + short_sleeve

    # Force exact public endpoints after floating-point construction.
    total_nav[0] = CFG.starting_nav
    total_nav[june_end] = june_peak_nav
    total_nav[-1] = july_end_nav

    intervention_level = june_peak_nav * (
        1.0 - CFG.intervention_drawdown_from_peak
    )
    crossing = np.flatnonzero(
        (np.arange(len(dates)) >= stress_start) & (total_nav <= intervention_level)
    )
    if crossing.size == 0:
        raise RuntimeError("The illustrative stress path did not breach intervention.")
    intervention_idx = int(crossing[0])

    return {
        "dates": dates,
        "long_sleeve": long_sleeve,
        "short_sleeve": short_sleeve,
        "total_nav": total_nav,
        "june_end": june_end,
        "stress_start": stress_start,
        "intervention_idx": intervention_idx,
        "final_idx": final_idx,
        "june_peak_nav": june_peak_nav,
        "july_end_nav": july_end_nav,
        "intervention_level": intervention_level,
    }


# ============================================================
# Illustrative one-week VaR and CVaR model
# ============================================================


def build_risk_metrics(
    dates: pd.DatetimeIndex,
    june_end: int,
    seed: int = CFG.random_seed + 100,
) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    n = len(dates)

    # Exposure scales with the gain rather than being cut after a huge run.
    progress = np.clip(np.arange(n) / max(june_end, 1), 0.0, 1.0)
    long_multiple = (
        CFG.start_long_multiple
        + (CFG.peak_long_multiple - CFG.start_long_multiple) * progress
    )
    short_multiple = (
        CFG.start_short_multiple
        + (CFG.peak_short_multiple - CFG.start_short_multiple) * progress
    )

    # Multivariate Student-t draws create fatter tails than a Gaussian VaR.
    correlation = np.array(
        [
            [1.0, CFG.normal_factor_correlation],
            [CFG.normal_factor_correlation, 1.0],
        ]
    )
    chol = np.linalg.cholesky(correlation)
    z = rng.normal(size=(CFG.monte_carlo_scenarios, 2)) @ chol.T
    chi_square = rng.chisquare(
        CFG.student_t_degrees_freedom,
        size=CFG.monte_carlo_scenarios,
    )
    t_draws = z / np.sqrt(
        chi_square[:, None] / CFG.student_t_degrees_freedom
    )

    # Normalize the t draws to unit standard deviation.
    t_draws /= np.sqrt(
        CFG.student_t_degrees_freedom
        / (CFG.student_t_degrees_freedom - 2.0)
    )

    one_week_long = (
        t_draws[:, 0] * CFG.long_daily_vol * np.sqrt(5.0)
    )
    one_week_short_basket = (
        t_draws[:, 1] * CFG.short_basket_daily_vol * np.sqrt(5.0)
    )

    var = np.empty(n, dtype=float)
    cvar = np.empty(n, dtype=float)

    for i in range(n):
        portfolio_return = (
            long_multiple[i] * one_week_long
            - short_multiple[i] * one_week_short_basket
        )
        loss = -portfolio_return
        threshold = np.quantile(loss, CFG.var_confidence)
        var[i] = threshold
        cvar[i] = loss[loss >= threshold].mean()

    bad_week_stress = -(
        CFG.peak_long_multiple * CFG.stress_long_return
        - CFG.peak_short_multiple * CFG.stress_short_basket_return
        + CFG.stress_liquidity_and_financing
    )

    return {
        "long_multiple": long_multiple,
        "short_multiple": short_multiple,
        "gross_multiple": long_multiple + short_multiple,
        "var": var,
        "cvar": cvar,
        "bad_week_stress": np.full(n, bad_week_stress, dtype=float),
        "reported_july": np.full(n, -CFG.reported_july_drawdown, dtype=float),
    }


# ============================================================
# Build data and perform consistency checks
# ============================================================


history = build_nav_history()
dates = history["dates"]
long_sleeve = history["long_sleeve"]
short_sleeve = history["short_sleeve"]
total_nav = history["total_nav"]
june_end = int(history["june_end"])
stress_start = int(history["stress_start"])
intervention_idx = int(history["intervention_idx"])
final_idx = int(history["final_idx"])
june_peak_nav = float(history["june_peak_nav"])
july_end_nav = float(history["july_end_nav"])
intervention_level = float(history["intervention_level"])

risk = build_risk_metrics(dates, june_end)
long_multiple = risk["long_multiple"]
short_multiple = risk["short_multiple"]
gross_multiple = risk["gross_multiple"]
weekly_var = risk["var"]
weekly_cvar = risk["cvar"]
bad_week_stress = risk["bad_week_stress"]
reported_july = risk["reported_july"]

public_13f_total = (
    CFG.common_stock_value + CFG.call_underlying_value + CFG.put_underlying_value
)
public_long_plus_calls = CFG.common_stock_value + CFG.call_underlying_value
public_delta_long = (
    CFG.common_stock_value + CFG.assumed_call_delta * CFG.call_underlying_value
)
public_delta_short = abs(CFG.assumed_put_delta) * CFG.put_underlying_value

# Sanity checks
if not np.isclose(public_13f_total, 13_676_657_577.0):
    raise RuntimeError("13F component values no longer sum to the filed total.")
if not np.isclose(total_nav[june_end], 539.0):
    raise RuntimeError("June endpoint must equal a +439% reported return.")
if not np.isclose(total_nav[-1] / total_nav[june_end] - 1.0, -0.67):
    raise RuntimeError("July endpoint must equal a -67% reported drawdown.")
if not weekly_cvar[june_end] > weekly_var[june_end]:
    raise RuntimeError("CVaR must exceed VaR.")
if not bad_week_stress[-1] > CFG.intervention_drawdown_from_peak:
    raise RuntimeError("The bad-week stress must breach the intervention threshold.")


# ============================================================
# Figure layout
# ============================================================


fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.61, 0.39],
    horizontal_spacing=0.11,
    specs=[[{"type": "xy"}, {"type": "bar"}]],
    subplot_titles=(
        "Attributed fund equity",
        "One-week loss estimates at current exposure",
    ),
)

initial_n = 1


# ============================================================
# Trace builders
# ============================================================


def total_nav_trace(n_points: int) -> go.Scatter:
    visible = total_nav[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=TOTAL_GREEN, width=4.5),
        text=endpoint_text(f"  Total {visible[-1]:.0f}", n_points),
        textposition="middle right",
        textfont=dict(color=TOTAL_GREEN, size=11),
        name="Total fund NAV",
        legendgroup="nav",
        showlegend=True,
        customdata=np.column_stack(
            [
                long_multiple[:n_points],
                short_multiple[:n_points],
                gross_multiple[:n_points],
                weekly_var[:n_points],
                weekly_cvar[:n_points],
            ]
        ),
        hovertemplate=(
            "<b>Illustrative fund NAV</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "NAV index: %{y:.1f}<br>"
            "Long exposure: %{customdata[0]:.2f}x NAV<br>"
            "Short exposure: %{customdata[1]:.2f}x NAV<br>"
            "Gross exposure: %{customdata[2]:.2f}x NAV<br>"
            "1-week 99% VaR: %{customdata[3]:.1%}<br>"
            "1-week 99% CVaR: %{customdata[4]:.1%}"
            "<extra></extra>"
        ),
    )


def sleeve_trace(
    sleeve_name: str,
    values: np.ndarray,
    color: str,
    n_points: int,
) -> go.Scatter:
    visible = values[:n_points]
    return go.Scatter(
        x=dates[:n_points],
        y=visible,
        mode="lines+text",
        line=dict(color=color, width=2.8),
        text=endpoint_text(f"  {sleeve_name} {visible[-1]:.0f}", n_points),
        textposition="middle right",
        textfont=dict(color=color, size=10),
        name=f"{sleeve_name} sleeve equity",
        legendgroup=sleeve_name.lower(),
        showlegend=True,
        hovertemplate=(
            f"<b>{sleeve_name} sleeve attribution</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Equity index contribution: %{y:.1f}"
            "<extra></extra>"
        ),
    )


def intervention_marker(n_points: int) -> go.Scatter:
    if n_points <= intervention_idx:
        x, y, text = [], [], []
    else:
        x = [dates[intervention_idx]]
        y = [total_nav[intervention_idx]]
        text = ["  Financing breaks"]

    return go.Scatter(
        x=x,
        y=y,
        mode="markers+text",
        marker=dict(
            color=STRESS_RED,
            size=14,
            symbol="x",
            line=dict(color=MARKER_WHITE, width=1.5),
        ),
        text=text,
        textposition="bottom right",
        textfont=dict(color=STRESS_RED, size=11),
        name="Illustrative intervention",
        showlegend=False,
        hovertemplate=(
            "<b>Illustrative prime-broker intervention</b><br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "NAV index: %{y:.1f}<br>"
            f"Threshold: {intervention_level:.1f}"
            "<extra></extra>"
        ),
    )


def risk_bar_trace(frame_position: int) -> go.Bar:
    # Keep future outcomes hidden until the relevant phase arrives.
    stress_value = (
        bad_week_stress[frame_position]
        if frame_position >= stress_start
        else 0.0
    )
    reported_value = (
        reported_july[frame_position]
        if frame_position >= final_idx
        else 0.0
    )

    values = [
        weekly_var[frame_position],
        weekly_cvar[frame_position],
        stress_value,
        reported_value,
    ]
    labels = ["99% VaR", "99% CVaR", "Bad-week stress", "Reported July"]
    bar_text = [pct(v) if v > 0 else "" for v in values]

    return go.Bar(
        x=values,
        y=labels,
        orientation="h",
        marker=dict(
            color=[VAR_YELLOW, CVAR_ORANGE, STRESS_RED, REPORTED_PURPLE],
            line=dict(color="rgba(255,255,255,0.22)", width=1),
        ),
        text=bar_text,
        textposition="outside",
        textfont=dict(color=OFF_WHITE, size=12),
        cliponaxis=False,
        name="Loss estimate",
        showlegend=False,
        customdata=np.array(
            [
                gross_multiple[frame_position],
                gross_multiple[frame_position],
                CFG.peak_long_multiple + CFG.peak_short_multiple,
                np.nan,
            ]
        ),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Loss: %{x:.1%} of NAV<br>"
            "Gross exposure: %{customdata:.2f}x NAV"
            "<extra></extra>"
        ),
    )


def traces_for_position(frame_position: int) -> list[go.BaseTraceType]:
    n_points = frame_position + 1
    return [
        total_nav_trace(n_points),
        sleeve_trace("Long", long_sleeve, LONG_BLUE, n_points),
        sleeve_trace("Short", short_sleeve, SHORT_ORANGE, n_points),
        intervention_marker(n_points),
        risk_bar_trace(frame_position),
    ]


# ============================================================
# Figure title
# ============================================================


FIGURE_TITLE = (
    "Situational Awareness LP: How a High-Return Long/Short Book Can Still Blow Up"
)


# ============================================================
# Initial traces
# ============================================================


fig.add_trace(total_nav_trace(initial_n), row=1, col=1)
fig.add_trace(sleeve_trace("Long", long_sleeve, LONG_BLUE, initial_n), row=1, col=1)
fig.add_trace(sleeve_trace("Short", short_sleeve, SHORT_ORANGE, initial_n), row=1, col=1)
fig.add_trace(intervention_marker(initial_n), row=1, col=1)
fig.add_trace(risk_bar_trace(0), row=1, col=2)


# ============================================================
# Static guides and event shading
# ============================================================


fig.add_hline(
    y=CFG.starting_nav,
    line=dict(color=BASELINE, width=1.2, dash="dot"),
    opacity=0.80,
    row=1,
    col=1,
)

fig.add_hline(
    y=intervention_level,
    line=dict(color=STRESS_RED, width=2.0, dash="dash"),
    opacity=0.88,
    row=1,
    col=1,
)

fig.add_vrect(
    x0=dates[stress_start],
    x1=dates[final_idx],
    fillcolor="rgba(255,77,90,0.10)",
    line_width=0,
    layer="below",
    row=1,
    col=1,
)

for event_idx in [june_end, stress_start, intervention_idx]:
    fig.add_vline(
        x=dates[event_idx],
        line=dict(color=BASELINE, width=1.2, dash="dot"),
        opacity=0.75,
        row=1,
        col=1,
    )

# Risk-limit line on the horizontal bar panel.
fig.add_vline(
    x=CFG.one_week_risk_limit,
    line=dict(color=STRESS_RED, width=2.0, dash="dash"),
    opacity=0.90,
    row=1,
    col=2,
)


# ============================================================
# Animation frames and slider
# ============================================================


frames: list[go.Frame] = []
slider_steps: list[dict[str, object]] = []

frame_positions = list(range(0, len(dates), CFG.frame_stride))
if frame_positions[-1] != final_idx:
    frame_positions.append(final_idx)

for key_idx in [june_end, stress_start, intervention_idx, final_idx]:
    if key_idx not in frame_positions:
        frame_positions.append(key_idx)
frame_positions = sorted(set(frame_positions))

for frame_position in frame_positions:
    frame_name = f"frame_{frame_position}"
    frame_data = traces_for_position(frame_position)

    frames.append(
        go.Frame(
            name=frame_name,
            data=frame_data,
            traces=list(range(len(frame_data))),
        )
    )

    slider_steps.append(
        {
            "label": dates[frame_position].strftime("%b %d"),
            "method": "animate",
            "args": [
                [frame_name],
                {
                    "frame": {"duration": 0, "redraw": True},
                    "mode": "immediate",
                    "transition": {"duration": 0},
                },
            ],
        }
    )

fig.frames = frames


# ============================================================
# Axes and layout
# ============================================================


fig.update_xaxes(
    AXIS_STYLE,
    row=1,
    col=1,
    range=[dates[0], dates[-1]],
    title_text="2026 trading date",
    title_standoff=20,
)
fig.update_yaxes(
    AXIS_STYLE,
    row=1,
    col=1,
    range=[0.0, june_peak_nav * 1.13],
    title_text="NAV index / attributed sleeve equity",
    separatethousands=True,
)

fig.update_xaxes(
    AXIS_STYLE,
    row=1,
    col=2,
    range=[0.0, 0.76],
    title_text="Potential one-week loss as % of NAV",
    tickformat=".0%",
    title_standoff=20,
)
fig.update_yaxes(
    dict(AXIS_STYLE),
    row=1,
    col=2,
    range=[3.85, -0.75],
    title_text="",
)

fig.update_layout(
    title=dict(
        text=FIGURE_TITLE,
        x=0.5,
        font=dict(color=OFF_WHITE),
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=CFG.figure_width,
    margin=dict(t=115, b=185, r=95, l=90),
    hovermode="closest",
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.16,
        yanchor="top",
        font=dict(color=OFF_WHITE),
        traceorder="normal",
    ),
    updatemenus=[
        {
            "type": "buttons",
            "direction": "left",
            "showactive": False,
            "x": 0.10,
            "xanchor": "right",
            "y": -0.315,
            "yanchor": "top",
            "pad": {"r": 10, "t": 38},
            "buttons": [
                {
                    "label": "▶ Play",
                    "method": "animate",
                    "args": [
                        None,
                        {
                            "frame": {
                                "duration": CFG.frame_duration_ms,
                                "redraw": True,
                            },
                            "transition": {"duration": 0},
                            "fromcurrent": True,
                        },
                    ],
                },
                {
                    "label": "⏸ Pause",
                    "method": "animate",
                    "args": [
                        [None],
                        {
                            "frame": {"duration": 0, "redraw": True},
                            "mode": "immediate",
                            "transition": {"duration": 0},
                        },
                    ],
                },
            ],
        }
    ],
    sliders=[
        {
            "active": 0,
            "x": 0.15,
            "len": 0.82,
            "y": -0.315,
            "yanchor": "top",
            "currentvalue": {
                "prefix": "Date: ",
                "font": {"color": MUTED_WHITE},
            },
            "pad": {"b": 6, "t": 20},
            "steps": slider_steps,
        }
    ],
)

fig.update_annotations(font=dict(color=OFF_WHITE, size=14))

# Keep threshold labels close to their axes so they read as guides rather
# than floating callout boxes.
fig.add_annotation(
    x=dates[3],
    y=intervention_level,
    xref="x",
    yref="y",
    text="50% peak-loss intervention",
    showarrow=False,
    font=dict(color=STRESS_RED, size=11),
    xanchor="left",
    yanchor="bottom",
    yshift=6,
)

fig.add_annotation(
    x=CFG.one_week_risk_limit,
    y=3.62,
    xref="x2",
    yref="y2",
    text="25% risk limit",
    showarrow=False,
    font=dict(color=STRESS_RED, size=11),
    xanchor="left",
    yanchor="middle",
    xshift=4,
)


---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary** 
  - A casino is calm because its edge is fixed: $\text{Casino Edge} = -\mathbb{E}[\text{Net Payout}]$. Players feel every swing; the house only needs sizing and survival so it never blows out. In the zero-sum simulation, player wealth drifts to ruin while casino wealth is the exact mirror of aggregate losses
  - Traders are not the house. Their edge is not a constant; it lives inside a policy $\pi(a_t \mid s_t; \theta)$ whose parameters encode beliefs, fear, and learned behavior. Loss aversion and related biases push decisions away from optimal risk-taking — same pressure as any profession where you have to step up to the plate
  - Handing trades to an algorithm,  = f_{\text{model}}(s_t; \phi)$, does not remove emotion. Model choice and parameterization are still human. A fixed mean-reversion rule can survive early regime shifts and still fail when the market trends against its stale assumptions
  - Every modeled problem is a specification-and-parameter problem: $\pi(a_t \mid s_t; \theta) \implies f_{\text{model}}(a_t \mid s_t; \phi(\theta))$. The real world is non-stationary, so you always need to update — the only question is when, and how aggressively
  - Bottom line: you cannot remove emotion from trading because someone is always steering the ship. Survival comes from qualitative Bayesian updating of the policy — daily work on when to trust the model, when to revise it, and how hard to turn the dial

###### ______________________________________________________________________________________________________________________________________

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System


---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$